In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

senate_donations_2020 = pd.read_csv('../data/senate_general_indiv20.csv', dtype=str)

senate_donations_2020['TRANSACTION_AMT'] = senate_donations_2020['TRANSACTION_AMT'].astype(int)

senate_donations_2020.head()

,CMTE_ID,IMAGE_NUM,TRANSACTION_TP,NAME,CITY,STATE,ZIP_CODE,EMPLOYER,OCCUPATION,TRANSACTION_DT,TRANSACTION_AMT,CAND_ID,CAND_NAME,CAND_PTY_AFFILIATION,CAND_OFFICE_ST
0,C00647842,202010089285059826,15,"LOMBARDO, ROBERT",MANCHESTER,NH,03102,ELECTRONICS FOR IMAGING INC,TRADE COMPLIANCE MANAGER,2020-09-08,275,S0NH00300,"O'DONNELL, JUSTIN F",LIB,NH
1,C00716340,202010089285059649,15,"FLOYD, WILLIAM",EVANSTON,IL,60201,RETIRED,RETIRED,2020-09-10,500,S0IL00543,"CURRAN, MARK",REP,IL
2,C00716340,202010089285059651,15,"GULLY, MICHAEL",QUINCY,IL,62305,GULLY TRANSPORTATION,TRUCK LINE EXECUTIVE,2020-09-07,1000,S0IL00543,"CURRAN, MARK",REP,IL
3,C00716340,202010089285059651,15,"GULLY, MICHAEL",QUINCY,IL,62305,GULLY TRANSPORTATION,TRUCK LINE EXECUTIVE,2020-09-07,250,S0IL00543,"CURRAN, MARK",REP,IL
4,C00716340,202010089285059661,15,"O'BRIEN, KATHLEEN",LAKE FOREST,IL,60045,SELF,ATTORNEY,2020-09-02,250,S0IL00543,"CURRAN, MARK",REP,IL


In [2]:
# top fundraisers
senate_donations_2020.groupby('CAND_NAME').agg({'TRANSACTION_AMT':'sum'}).sort_values(by='TRANSACTION_AMT', ascending=False).head(20)

,TRANSACTION_AMT
CAND_NAME,
"GRAHAM, LINDSEY O.",40373110
"MCSALLY, MARTHA",26866551
"HARRISON, JAIME",24736723
"KELLY, MARK",23736714
"GIDEON, SARA",23309218
"MCGRATH, AMY",21988664
"BULLOCK, STEVE",21623416
"CUNNINGHAM, CAL",21251775
"HICKENLOOPER, JOHN W.",16651798


In [3]:
# top fundraisers
senate_donations_2020.groupby(['CAND_NAME', 'STATE']).agg({'TRANSACTION_AMT':'sum'}).sort_values(by='TRANSACTION_AMT', ascending=False).head(20)

,,TRANSACTION_AMT
CAND_NAME,STATE,
"HARRISON, JAIME",CA,5834052
"BULLOCK, STEVE",CA,5785181
"KELLY, MARK",CA,5776877
"GIDEON, SARA",CA,5772067
"MCGRATH, AMY",CA,5343582
"MCSALLY, MARTHA",AZ,4914584
"GRAHAM, LINDSEY O.",CA,4909875
"CUNNINGHAM, CAL",CA,4903251
"CORNYN, JOHN SEN",TX,4716322


In [4]:
# mark each donation as in-state or out-of-state
senate_donations_2020['IN_STATE'] = (senate_donations_2020['STATE'] == senate_donations_2020['CAND_OFFICE_ST'])

senate_donations_2020[['CAND_NAME', 'CAND_OFFICE_ST', 'STATE', 'IN_STATE']].head(10)

,CAND_NAME,CAND_OFFICE_ST,STATE,IN_STATE
0,"O'DONNELL, JUSTIN F",NH,NH,True
1,"CURRAN, MARK",IL,IL,True
2,"CURRAN, MARK",IL,IL,True
3,"CURRAN, MARK",IL,IL,True
4,"CURRAN, MARK",IL,IL,True
5,"ERNST, JONI K",IA,IA,True
6,"ERNST, JONI K",IA,IA,True
7,"ERNST, JONI K",IA,IA,True
8,"ERNST, JONI K",IA,IA,True
9,"ERNST, JONI K",IA,IA,True


In [5]:
# percent of in-state donations

total_sum = senate_donations_2020['TRANSACTION_AMT'].sum()
total_count = len(senate_donations_2020)

display(senate_donations_2020.groupby('IN_STATE').agg({'TRANSACTION_AMT':'sum'}) / total_sum)

display(senate_donations_2020.groupby('IN_STATE').agg({'TRANSACTION_AMT':'count'}) / total_count)

,TRANSACTION_AMT
IN_STATE,
False,0.811609
True,0.188391


,TRANSACTION_AMT
IN_STATE,
False,0.830389
True,0.169611


So roughly 80% of all campaign donations are out-of-state, by both count and amount. Surprising.

In [6]:
in_state_donations = senate_donations_2020[senate_donations_2020['IN_STATE'] == True]
out_of_state_donations = senate_donations_2020[senate_donations_2020['IN_STATE'] == False]

In [7]:
out_of_state_donations.groupby('STATE').agg({'TRANSACTION_AMT':'sum'}).sort_values(by='TRANSACTION_AMT', ascending=False).head(10)

,TRANSACTION_AMT
STATE,
CA,68305380
NY,31809380
FL,21172643
TX,19849052
MA,16823225
WA,12709640
IL,12049605
VA,10373299
MD,8915649


In [8]:
# # want to extract data on whether the candidate won or not
# # will ask an LLM to help me

# candidates_2020 = senate_donations_2020[['CAND_NAME', 'CAND_OFFICE_ST']].drop_duplicates()
# candidates_2020.to_csv('candidates_2020.csv', index=False)

In [9]:
# # by the way I'll do it for 2022 also
# senate_donations_2022 = pd.read_csv('../data/senate_general_indiv22.csv', dtype=str)
# candidates_2022 = senate_donations_2022[['CAND_NAME', 'CAND_OFFICE_ST']].drop_duplicates()
# candidates_2022.to_csv('candidates_2022.csv', index=False)

# Build dataframe

In [10]:
# Pivot to get in-state and out-of-state amounts side by side
donations_by_candidate = (
    senate_donations_2020
    .groupby(['CAND_NAME', 'CAND_OFFICE_ST', 'IN_STATE', 'CAND_PTY_AFFILIATION'])['TRANSACTION_AMT']
    .sum()
    .unstack('IN_STATE', fill_value=0)
    .rename(columns={False: 'amt_out_state', True: 'amt_in_state'})
    .reset_index()
)

donations_by_candidate['total_amt'] = (
    donations_by_candidate['amt_in_state'] + donations_by_candidate['amt_out_state']
)
donations_by_candidate['pct_in_state'] = (
    donations_by_candidate['amt_in_state'] / donations_by_candidate['total_amt']
)

donations_by_candidate

IN_STATE,CAND_NAME,CAND_OFFICE_ST,CAND_PTY_AFFILIATION,amt_out_state,amt_in_state,total_amt,pct_in_state
0,"AHLERS, DAN",SD,DEM,21125,44768,65893,0.679404
1,"ALEXANDER, LAMAR",TN,REP,37300,0,37300,0.000000
2,"AYYADURAI, SHIVA DR",MA,REP,13361,5925,19286,0.307218
3,"BAER, DAN",CO,DEM,277685,143950,421635,0.341409
4,"BARRON, STEPHEN BRADLEY",KY,LIB,43997,62627,106624,0.587363
...,...,...,...,...,...,...,...
140,"WHITFIELD, DANIEL ALLEN",AR,IND,4750,5139,9889,0.519668
141,"WILLIAMS, ANGELA",CO,DEM,0,500,500,1.000000
142,"WINFIELD, RICHARD DIEN",GA,DEM,11200,2775,13975,0.198569
143,"WITZKE, LAUREN ELENA",DE,REP,97548,30414,127962,0.237680


In [11]:
results_2020 = pd.read_csv('candidate_results_2020.csv')

# join
donations_by_candidate = pd.merge(donations_by_candidate,
                                  results_2020[['CAND_NAME', 'OUTCOME', 'STAGE', 'NOTES']],
                                  on='CAND_NAME',
                                  how='left')

donations_by_candidate.head()

,CAND_NAME,CAND_OFFICE_ST,CAND_PTY_AFFILIATION,amt_out_state,amt_in_state,total_amt,pct_in_state,OUTCOME,STAGE,NOTES
0,"AHLERS, DAN",SD,DEM,21125,44768,65893,0.679404,Lost,General,Won D primary; lost general to Rounds
1,"ALEXANDER, LAMAR",TN,REP,37300,0,37300,0.000000,NaN,NaN,Incumbent who retired; did not seek re-electio...
2,"AYYADURAI, SHIVA DR",MA,REP,13361,5925,19286,0.307218,Lost,Primary,Lost Republican primary to O'Connor (39.4%)
3,"BAER, DAN",CO,DEM,277685,143950,421635,0.341409,Lost,Primary,Lost Democratic primary to Hickenlooper
4,"BARRON, STEPHEN BRADLEY",KY,LIB,43997,62627,106624,0.587363,Lost,Primary,Lost Democratic primary to McGrath


In [18]:
# define a serious candidate as someone who makes it to the general and raises at least 100k

donations_by_candidate['top_in_state_party'] = (
    donations_by_candidate['STAGE'] == 'General'  # <-- was: groupby transform max
)

serious = donations_by_candidate[
    (donations_by_candidate['total_amt'] >= 100_000) |
    (donations_by_candidate['top_in_state_party'])
].copy()

print(f"{len(serious)} / {len(donations_by_candidate)} candidates kept")
serious

101 / 145 candidates kept


,CAND_NAME,CAND_OFFICE_ST,CAND_PTY_AFFILIATION,amt_out_state,amt_in_state,total_amt,pct_in_state,OUTCOME,STAGE,NOTES,top_in_state_party
0,"AHLERS, DAN",SD,DEM,21125,44768,65893,0.679404,Lost,General,Won D primary; lost general to Rounds,True
3,"BAER, DAN",CO,DEM,277685,143950,421635,0.341409,Lost,Primary,Lost Democratic primary to Hickenlooper,False
4,"BARRON, STEPHEN BRADLEY",KY,LIB,43997,62627,106624,0.587363,Lost,Primary,Lost Democratic primary to McGrath,False
6,"BEN DAVID, MERAV",WY,DEM,20559,2900,23459,0.123620,Lost,General,Won D primary; lost general to Lummis,True
7,"BOLDUC, DONALD C.",NH,REP,149514,44144,193658,0.227948,Lost,Primary,Lost Republican primary to Messner (43% vs 51%),False
...,...,...,...,...,...,...,...,...,...,...,...
135,"WARNER, MARK ROBERT",VA,DEM,488522,468059,956581,0.489304,Won,General,Incumbent Democrat; won re-election (nominated...,True
137,"WATERS, ALLEN",RI,REP,3105,100,3205,0.031201,Lost,General,Won R primary; lost general to Reed,True
138,"WENSTRUP, PETER",LA,DEM,63767,11660,75427,0.154587,Lost,General,Lost Louisiana jungle primary/general to Cassidy,True
140,"WHITFIELD, DANIEL ALLEN",AR,IND,4750,5139,9889,0.519668,Lost,General,Won D primary; lost general to Cotton,True


In [19]:
# I'm just doing linear regression on the probabilities for interpretability (hate log-odds)

import statsmodels.formula.api as smf

# OUTCOME is 'Won'/'Lost' — create binary
serious['won'] = (serious['OUTCOME'] == 'Won').astype(int)

model = smf.logit('won ~ total_amt + pct_in_state', data=serious).fit()
model.summary()

Optimization terminated successfully.
         Current function value: 0.625670
         Iterations 5


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                    won   No. Observations:                  101
Model:                          Logit   Df Residuals:                       98
Method:                           MLE   Df Model:                            2
Date:                Thu, 14 May 2026   Pseudo R-squ.:                 0.02052
Time:                        23:03:19   Log-Likelihood:                -63.193
converged:                       True   LL-Null:                       -64.517
Covariance Type:            nonrobust   LLR p-value:                    0.2661
================================================================================
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       -0.4455      0.455     -0.980      0.327      -1.336       0.445
total_amt      2.36e-08      3e-08      0.786      0.432   -3.52e-08    8.24e-08
pct_in_state    -0.8115      0.851     -0.954      0.340      -2.479       0.856
================================================================================
"""